# Top Brass -  Example 5.1 in Rardin (1998)

Top Brass Trophy Company makes large championship trophies for youth athletic leagues. At the moment, they are planning production for fall sports: football and soccer. Each football trophy has a wood base, an engraved plaque, a large brass football on top, and returns \\$12 in profit. Soccer trophies are similar (a brass ball, a wood base, and an engraved plaque) except that a brass soccer ball is on top, and the unit profit is only \\$9. Since the football has an asymmetric shape, its base requires 4 board feet of wood; the soccer base requires only 2 board feet. At the moment there are 1000 brass footballs in stock, 1500 soccer balls, 1750 plaques, and 4800 board feet of wood. What trophies should be produced from these supplies to maximize total profit assuming that all that are made can be sold?

## The Top Brass Model

In [10]:
# always specify which packages you're going to use
using JuMP, HiGHS

#create a new model object
m = Model()

# we need variables for football trophies and soccer trophies
# format is (<model name>, <variable name>). we can optionally
# include bounds on each variable.
@variable(m, ft >= 0)
@variable(m, st >= 0)

# objective is to maximize profit
# format is (<model name>, <Max or Min>, <algebraic function>)
@objective(m, Max, 12*ft + 9*st)

# constraint on the wood available
# format is (<model name>, <constraint name>, <algebraic constraint>)
@constraint(m, wood_con, 4ft + 2st <= 4800)

#constraint on the plaques available
@constraint(m, plaque_con, ft + st <= 1750)

# constraints on brass footballs, soccerballs available
@constraint(m, brass_football_con, ft <= 1000)
@constraint(m, brass_soccerball_con, st <= 1500)
; 
# like Matlab, a ";" (semicolon) supresses output

Now let's solve it and take a look at the values!

In [14]:
println("Time to solve this model using HiGHS: ")

# specify the solver you want to use to solve Model m
set_optimizer(m, HiGHS.Optimizer)

# if you want to supress the solver output, use "set_silent(<model_name>)"
# we'll leave it off so you can see what different solver output looks like
# set_silent(m)

# it's best practice to run your model once with full solver output so you can check 
# that Model status is "Optimal". 

# use the @time macro to measure the amount of time it takes to solve m
@time(optimize!(m))

println("Build ", value(ft), " football trophies.")
println("Build ", value(st), " soccer trophies.")
println("Total profit will be \$", objective_value(m))

Time to solve this model using HiGHS: 
Running HiGHS 1.10.0 (git hash: fd8665394e): Copyright (c) 2025 HiGHS under MIT licence terms
LP   has 4 rows; 2 cols; 6 nonzeros
Coefficient ranges:
  Matrix [1e+00, 4e+00]
  Cost   [9e+00, 1e+01]
  Bound  [0e+00, 0e+00]
  RHS    [1e+03, 5e+03]
Presolving model
2 rows, 2 cols, 4 nonzeros  0s
2 rows, 2 cols, 4 nonzeros  0s
Presolve : Reductions: rows 2(-2); columns 2(-0); elements 4(-2)
Solving the presolved LP
Using EKK dual simplex solver - serial
  Iteration        Objective     Infeasibilities num(sum)
          0     0.0000000000e+00 Ph1: 0(0) 0s
          2    -1.7700000000e+04 Pr: 0(0) 0s
Solving the original LP from the solution after postsolve
Model status        : Optimal
Simplex   iterations: 2
Objective value     :  1.7700000000e+04
Relative P-D gap    :  0.0000000000e+00
HiGHS run time      :          0.00
  0.007838 seconds (334 allocations: 15.766 KiB)
Build 650.0 football trophies.
Build 1100.0 soccer trophies.
Total profit will be

Let's see what happens with a solver that isn't built to solve this type of model!

In [15]:
using ECOS

println("Time to solve this model using ECOS: ")
set_optimizer(m, ECOS.Optimizer)

@time(optimize!(m))

println("Build ", value(ft), " football trophies.")
println("Build ", value(st), " soccer trophies.")
println("Total profit will be \$", objective_value(m))

# some solvers (including ECOS), output a lot of information along with the solution.
# it can be helpful to explicitly print some desired solution components, as we've done here.

Time to solve this model using ECOS: 
  0.018696 seconds (2.32 k allocations: 112.766 KiB)
Build 649.9999988187327 football trophies.
Build 1100.000000607335 soccer trophies.
Total profit will be $17699.99999129081

ECOS 2.0.8 - (C) embotech GmbH, Zurich Switzerland, 2012-15. Web: www.embotech.com/ECOS

It     pcost       dcost      gap   pres   dres    k/t    mu     step   sigma     IR    |   BT
 0  -1.586e+04  -3.576e+04  +1e+04  3e-06  3e-01  1e+00  1e+03    ---    ---    1  1  - |  -  - 
 1  -1.765e+04  -1.891e+04  +7e+02  2e-07  2e-02  1e+01  1e+02  0.9369  1e-02   0  0  0 |  0  0
 2  -1.769e+04  -1.773e+04  +2e+01  6e-09  7e-04  1e+00  3e+00  0.9820  1e-02   0  0  0 |  0  0
 3  -1.770e+04  -1.770e+04  +2e-01  6e-11  8e-06  1e-02  3e-02  0.9890  1e-04   1  0  0 |  0  0
 4  -1.770e+04  -1.770e+04  +2e-03  7e-13  9e-08  1e-04  4e-04  0.9890  1e-04   1  0  0 |  0  0
 5  -1.770e+04  -1.770e+04  +3e-05  8e-15  1e-09  2e-06  4e-06  0.9890  1e-04   1  0  0 |  0  0

OPTIMAL (within feasto